# Setup env

In [6]:
import pandas as pd
from pathlib import Path
import sys

# Add the src package folder to Python path so imdb_utils can be found
src_path = Path.cwd()
if not (src_path / "src").exists():
    project_root = next((parent for parent in src_path.parents if (parent / "src").exists()), src_path)
else:
    project_root = src_path
sys.path.insert(0, str(project_root / "src"))
from imdb_utils import utils

In [7]:
#  Constants
data_path = 'C:\\Users\\andre\\Downloads\\imdb'

# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

Your project source root is: c:\Users\andre\Python\dw\imbd


# Read raw sources

In [13]:
# Read raw data
title_basics = pd.read_csv(f'{data_path}\\title.basics.tsv\\data.csv', sep='\t', na_values='\\N')
title_basics = title_basics[title_basics['titleType'].isin(['tvSeries']) & title_basics['isAdult'] == 0]
# name_basics = pd.read_csv(f'{data_path}name.basics.tsv/data.tsv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')
title_principals = pd.read_csv(f'{data_path}\\title.principals.tsv\\data.csv', sep='\t', na_values='\\N')

title_episode = pd.read_csv(
    f'{data_path}\\title.episode.tsv\\data.tsv',
    sep='\t',
    na_values='\\N',
    dtype={'seasonNumber': 'Int16', 'episodeNumber': 'Int16'},
    low_memory=False
)

title_ratings = pd.read_csv(
    f'{data_path}\\title.ratings.tsv\\data.tsv',
    sep='\t',
    na_values='\\N'
)

C:\Users\andre\AppData\Local\Temp\ipykernel_21992\2810146440.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv(f'{data_path}\\title.basics.tsv\\data.csv', sep='\t', na_values='\\N')


# Dimensional tables

## dim_time

In [15]:
# Create dim_time covering the full range of IMDb data
#
# Sources:
#   generated — no raw file required (generate_series equivalent)
#
# Design notes:
#   - Grain: one row per calendar year (annual granularity matches startYear in title_basics)
#   - tv_era classifies each year into a qualitative television era for analytical grouping
#   - period provides a coarser grouping for decade-level analysis
#   - No measures — this is a pure descriptive dimension

ERA_BINS   = [0,     1979,         1999,          2012,           9999]
ERA_LABELS = ['Classic Era', 'Cable Era', 'Quality Era', 'Streaming Era']

PERIOD_BINS   = [0,     1999,          2009,          2019,    9999]
PERIOD_LABELS = ['Before 2000', '2000–2009', '2010–2019', '2020+']

dim_time = pd.DataFrame({'year': range(1900, 2031)}, dtype='int16')
dim_time['decade'] = ((dim_time['year'] // 10) * 10).astype('int16')
dim_time['tv_era'] = pd.cut(dim_time['year'], bins=ERA_BINS,    labels=ERA_LABELS).astype(str)
dim_time['period'] = pd.cut(dim_time['year'], bins=PERIOD_BINS, labels=PERIOD_LABELS).astype(str)

dim_time = dim_time[['year', 'decade', 'tv_era', 'period']]

print(f"dim_time shape: {dim_time.shape}")
print(f"Sample:\n{dim_time.head(3)}")
print(f"\nTV eras:\n{dim_time['tv_era'].value_counts().sort_index()}")

# Write parquet file
dim_time.to_parquet(f'{project_src}\\data\\silver\\dim_time.parquet', compression='snappy', engine='pyarrow')

del dim_time

dim_time shape: (131, 4)
Sample:
   year  decade       tv_era       period
0  1900    1900  Classic Era  Before 2000
1  1901    1900  Classic Era  Before 2000
2  1902    1900  Classic Era  Before 2000

TV eras:
tv_era
Cable Era        20
Classic Era      80
Quality Era      13
Streaming Era    18
Name: count, dtype: int64


In [16]:
# Create a minimal filter dimension table
dim_filter = title_basics[['tconst', 'titleType', 'isAdult']].copy()
dim_filter = dim_filter[(dim_filter['titleType'] == 'tvSeries') & (dim_filter['isAdult'] == 0)]

# Add nconst by joining with title_principals
dim_filter = dim_filter.merge(
    title_principals[['tconst', 'nconst']],
    on='tconst',
    how='inner'
).drop_duplicates()

# Reorder columns
dim_filter = dim_filter[['nconst', 'tconst', 'titleType', 'isAdult']]

print(f"dim_filter shape: {dim_filter.shape}")
print(f"Sample:\n{dim_filter.head()}")

# Write parquet file
dim_filter.to_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet', compression='snappy', engine='pyarrow')

dim_filter shape: (1133198, 4)
Sample:
      nconst     tconst titleType  isAdult
0  nm0345300  tt0032557  tvSeries      0.0
1  nm0430460  tt0032557  tvSeries      0.0
2  nm0580585  tt0032557  tvSeries      0.0
3  nm0186608  tt0032557  tvSeries      0.0
4  nm0279963  tt0032557  tvSeries      0.0


In [17]:
dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')
dim_filter

,nconst,tconst,titleType,isAdult
0,nm0345300,tt0032557,tvSeries,0.0
1,nm0430460,tt0032557,tvSeries,0.0
2,nm0580585,tt0032557,tvSeries,0.0
3,nm0186608,tt0032557,tvSeries,0.0
4,nm0279963,tt0032557,tvSeries,0.0
...,...,...,...,...
1133546,nm8011539,tt9916678,tvSeries,0.0
1133547,nm10749212,tt9916678,tvSeries,0.0
1133548,nm9613196,tt9916678,tvSeries,0.0
1133549,nm10783601,tt9916678,tvSeries,0.0


## dim_profession | bridge_profession_group

In [19]:
# Create dim_profession from unique professions in primaryProfession
# Extract all unique professions from comma-separated primaryProfession values

# Sources:
# name_basics

# 2. Create dim_profession (Vectorized unique extraction)
unique_profs = (
    name_basics['primaryProfession']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_profession = pd.DataFrame({'profession_nm': sorted(unique_profs)})
dim_profession['profession_id'] = [f'prf_{i+1}' for i in range(len(dim_profession))]


# 3. Create bridge_profession_group (Vectorized expansion)
# Explode the professions column into individual rows
bridge_profession_group = (
    name_basics[['nconst', 'primaryProfession']]
    .dropna(subset=['primaryProfession'])
    .assign(profession_nm=lambda x: x['primaryProfession'].str.split(','))
    .explode('profession_nm')
)

# 4. Map the IDs using Merge (Replaces the dictionary lookup loop)
bridge_profession_group = (
    bridge_profession_group
    .reset_index()
    .merge(dim_profession.reset_index(), on='profession_nm')
    [['nconst', 'profession_id']]
)

# Create profession_group_id for each unique combination of professions per person
bridge_profession_group['profession_group_id'] = 'p_grp_' + (bridge_profession_group.groupby('nconst').ngroup() + 1).astype(str)

# Load dim_filter and apply inner join to keep only data in dim_filter
bridge_profession_group = bridge_profession_group.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)
bridge_profession_group['weighting_factor_prf'] = 1 / bridge_profession_group.groupby('nconst')['profession_id'].transform('count')

# Write parquet files
bridge_profession_group.to_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet', compression='snappy', engine='pyarrow')
dim_profession.to_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet', compression='snappy', engine='pyarrow')


print(f"\nbridge_profession_group shape (after filtering): {bridge_profession_group.shape}")
print(f"Sample bridge data:\n{bridge_profession_group.head(10)}")

del bridge_profession_group
del dim_profession

NameError: name 'name_basics' is not defined

## dim_person

In [6]:
# Create dim_person from name_basics
# Load name_basics data

# Sources:
# name_basics
# bridge_profession_group
# dim_filter

bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
# dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# Select required columns for dim_person
dim_person = utils.select_cols(name_basics, ['nconst', 'primaryName', 'birthYear', 'deathYear'])

# Join with bridge_profssion_group to get profession_group_id
per_prfession = dim_person.merge(bridge_profession_group, on='nconst', how='left')
# Apply inner join with dim_filter to keep only participants in dim_filter
dim_person = dim_person.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)

# Remove duplicates if any and set nconst as index
dim_person = per_prfession.drop_duplicates(subset=['nconst'])
dim_person = dim_person[['nconst','primaryName', 'birthYear', 'deathYear', 'profession_group_id']]

print(f"dim_person shape (after filtering): {dim_person.shape}")
# Write parquet files
dim_person.to_parquet(f'{project_src}\\data\\silver\\dim_person.parquet', compression='snappy', engine='pyarrow')

del dim_person

dim_person shape (after filtering): (10777803, 5)


## dim_roles

In [4]:
# # Create dim_roles from title_principals and title_basics
# # Load title_principals and title_basics data

# Sources:
# title_principals
# title_basics
# dim_filter

# Final dim cols
cols_to_keep = ['role_id', 'nconst', 'tconst', 'titleType', 'category', 'job', 'characters']
dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# Join title_principals with title_basics to get titleType
dim_roles = title_principals.merge(title_basics[['tconst', 'titleType']], on='tconst', how='left')
# Apply inner join with dim_filter to keep only participants in dim_filter
dim_roles = dim_roles.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)
# Create a unique role_id combining nconst, tconst, and ordering
dim_roles['role_id'] = dim_roles['nconst'] + '_' + dim_roles['tconst'] + '_' + dim_roles['ordering'].astype(str)

# Select and reorder columns
dim_roles = utils.select_cols(dim_roles, cols_to_keep)

# Remove duplicates and set role_id as index
dim_roles = dim_roles.drop_duplicates(subset=['role_id'])

# Apply inner join with dim_filter to keep only roles in dim_filter
dim_roles = dim_roles.merge(
    dim_filter[['nconst', 'tconst']].drop_duplicates(),
    on=['nconst', 'tconst'],
    how='inner'
)

print(f"dim_roles shape (after filtering): {dim_roles.shape}")

# Write parquet files
dim_roles.to_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet', compression='snappy', engine='pyarrow')

del dim_roles

dim_roles shape (after filtering): (1133551, 7)


## dim_genre | bridge_genre_group

In [20]:
# 1. Load Data
# Sources:
# title_basics
# dim_filter

dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# 2. Create dim_genres (The Dimension Table)
# Extract unique genres using vectorized split and unique()
unique_genres = (
    title_basics['genres']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_genres = pd.DataFrame({'genre_nm': sorted(unique_genres)})
dim_genres['genre_id'] = [f'gen_{i+1}' for i in range(len(dim_genres))]

# 3. Create bridge_genres (The Bridge Table)
# Instead of a dictionary lookup, we use explode and then merge with dim_genres
bridge_genres = (
    title_basics[['tconst', 'genres']]
    .dropna(subset=['genres'])
    .assign(genre_nm=lambda x: x['genres'].str.split(','))
    .explode('genre_nm')
)

# Merge to get the genre_id (vectorized replacement for the dictionary lookup)
bridge_genres = (
    bridge_genres.reset_index()
    .merge(dim_genres.reset_index(), on='genre_nm')
    [['tconst', 'genre_id']]
)

# 4. Create genre_group_id
# This stays vectorized as before
bridge_genres['genre_group_id'] = (
    'g_grp_' + 
    (bridge_genres.groupby('tconst').ngroup() + 1).astype(str)
)
# Calculate the weight as 1 divided by the number of genres for that specific title
bridge_genres['weighting_factor_gen'] = 1 / bridge_genres.groupby('tconst')['genre_id'].transform('count')

# Apply inner join with dim_filter to keep only genres for titles in dim_filter
bridge_genres = bridge_genres.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

print(f"dim_genres shape: {dim_genres.shape}")
print(f"bridge_genres shape (after filtering): {bridge_genres.shape}")

# Write parquet files
dim_genres.to_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet', compression='snappy', engine='pyarrow')
bridge_genres.to_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet', compression='snappy', engine='pyarrow')

del dim_genres
del bridge_genres

dim_genres shape: (28, 2)
bridge_genres shape (after filtering): (225747, 4)


## dim_title_basic 

In [6]:
# Sources:
# title_basics
# bridge_genres
# dim_filter

bridge_genres = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')
# Select required columns for dim_title_basic
# dim_title_basic = title_basics[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes']].copy()
dim_title_basic = utils.select_cols(title_basics, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes'])

# Apply inner join with dim_filter to keep only titles in dim_filter
dim_title_basic = dim_title_basic.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

# Join with bridge_profssion_group to get profession_group_id
title_genre = dim_title_basic.join(bridge_genres.set_index('tconst'), on='tconst', how='left')
# title_genre = title_genre[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']].copy()
title_genre = utils.select_cols(title_genre, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id'])
# Remove duplicates if any and set tconst as index
dim_title_basic = title_genre.drop_duplicates(subset=['tconst'])

# Convert to numeric, forcing errors to NaN
dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')

# dim_title_basic = dim_title_basic[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']]
print(f"dim_title_basic shape (after filtering): {dim_title_basic.shape}")

#Write parquet files
dim_title_basic.to_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet', compression='snappy', engine='pyarrow')

del dim_title_basic

C:\Users\davib\AppData\Local\Temp\ipykernel_23240\1221509368.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')


dim_title_basic shape (after filtering): (175011, 9)


##  bridge_kt_group

In [25]:
# 1. Load data (Keeping your existing logic)
# Note: Using na_values='\\N' as discussed previously is perfect here

# Sources:
# title_basics
# name_basics 

name_basics = pd.read_csv(f'{data_path}\\name.basics.tsv\\data.csv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')

# 2. Vectorized Expansion (Replacing the two loops)
# We select only the columns we need, split the strings into lists, and "explode" them
bridge_kwn_titles = (
    name_basics[['nconst', 'knownForTitles']]
    .dropna(subset=['knownForTitles'])
    .assign(tconst=lambda x: x['knownForTitles'].str.split(','))
    .explode('tconst')
    [['tconst', 'nconst']]  # Reorder to match your desired output
)

# Apply inner join with dim_filter to keep only known titles for participants and titles in dim_filter
bridge_kwn_titles = bridge_kwn_titles.merge(
    dim_filter[['nconst', 'tconst']].drop_duplicates(),
    on=['nconst', 'tconst'],
    how='inner'
)

# 3. Vectorized ID Generation
# ngroup() is already vectorized, so we just keep this logic
bridge_kwn_titles['kwn_title_group_id'] = (
    'kwn_t_grp_' + 
    (bridge_kwn_titles.groupby('tconst').ngroup() + 1).astype(str)
)
bridge_kwn_titles['weighting_factor_grp'] = 1 / bridge_kwn_titles.groupby('nconst')['tconst'].transform('count')

# 4. Get Unique Titles Set (If still needed for other logic)
known_titles = set(bridge_kwn_titles['tconst'].unique())

print(f"bridge_kwn_titles shape (after filtering): {bridge_kwn_titles.shape}")
bridge_kwn_titles.to_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet', compression='snappy', engine='pyarrow')

del bridge_kwn_titles

bridge_kwn_titles shape (after filtering): (407324, 4)


## dim_series

In [27]:
# Create dim_series from title_basics
#
# Sources:
#   title_basics   — titleType, primaryTitle, originalTitle, startYear, endYear, isAdult
#   title_episode  — to compute num_seasons (MAX seasonNumber per parentTconst)
#   bridge_genre_group — to attach genre_group_id (many-to-many resolver)
#
# Design notes:
#   - Grain: one row per TV series (titleType = 'tvSeries')
#   - num_seasons is computed from title_episode, not stored in title_basics
#   - status is inferred: endYear IS NOT NULL → 'Concluded', else → 'On Air'
#   - genre_group_id links to bridge_genre_group to resolve the multi-valued genres field
#   - No measures — avg_rating and vote counts live exclusively in the fact table

bridge_genre_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')

# 1. Filter title_basics to tvSeries only, excluding adult content
series_mask = (
    (title_basics['titleType'] == 'tvSeries') &
    (title_basics['isAdult'].fillna(1).astype(int) == 0)
)
dim_series = title_basics.loc[series_mask].copy()

# 2. Compute num_seasons: MAX seasonNumber per series from title_episode
num_seasons = (
    title_episode
    .dropna(subset=['seasonNumber'])
    .groupby('parentTconst')['seasonNumber']
    .max()
    .reset_index()
    .rename(columns={'parentTconst': 'tconst', 'seasonNumber': 'num_seasons'})
)
dim_series = dim_series.merge(num_seasons, on='tconst', how='left')

# 3. Infer series status from endYear
dim_series['status'] = dim_series['endYear'].apply(
    lambda y: 'Concluded' if pd.notna(y) else 'On Air'
)

# 4. Attach genre_group_id (one genre_group_id per tconst — drop_duplicates ensures 1:1)
genre_lookup = bridge_genre_group[['tconst', 'genre_group_id']].drop_duplicates(subset=['tconst'])
dim_series = dim_series.merge(genre_lookup, on='tconst', how='left')

# 5. Select and rename final columns
dim_series = utils.select_cols(dim_series, [
    'tconst', 'primaryTitle', 'originalTitle',
    'startYear', 'endYear', 'num_seasons',
    'status', 'isAdult', 'genre_group_id'
])
dim_series = dim_series.rename(columns={
    'primaryTitle':  'primary_title',
    'originalTitle': 'original_title',
    'startYear':     'start_year',
    'endYear':       'end_year',
    'isAdult':       'is_adult',
})

# 6. Type coercions
dim_series['start_year']   = pd.to_numeric(dim_series['start_year'],   errors='coerce').astype('Int16')
dim_series['end_year']     = pd.to_numeric(dim_series['end_year'],     errors='coerce').astype('Int16')
dim_series['num_seasons']  = pd.to_numeric(dim_series['num_seasons'],  errors='coerce').astype('Int16')
dim_series['is_adult']     = dim_series['is_adult'].map({'1': True, '0': False, 1: True, 0: False}).fillna(False)

dim_series = dim_series.drop_duplicates(subset=['tconst']).reset_index(drop=True)

print(f"dim_series shape: {dim_series.shape}")
print(f"Status breakdown:\n{dim_series['status'].value_counts()}")
print(f"Sample:\n{dim_series.head(3)}")

# Write parquet file
dim_series.to_parquet(f'{project_src}\\data\\silver\\dim_series.parquet', compression='snappy', engine='pyarrow')

del dim_series
del bridge_genre_group
del genre_lookup
del num_seasons

dim_series shape: (201726, 9)
Status breakdown:
status
On Air       142532
Concluded     59194
Name: count, dtype: int64
Sample:
      tconst                primary_title               original_title  \
0  tt0032557             The Green Archer             The Green Archer   
1  tt0035599  Voice of Firestone Televues  Voice of Firestone Televues   
2  tt0035803     The German Weekly Review     Die Deutsche Wochenschau   

   start_year  end_year  num_seasons     status  is_adult genre_group_id  
0        1940      <NA>            1     On Air     False    g_grp_29229  
1        1943      1947         <NA>  Concluded     False            NaN  
2        1940      1945         <NA>  Concluded     False    g_grp_32221  


## dim_season

In [28]:
# Create dim_season from title_episode
#
# Sources:
#   title_episode  — parentTconst, seasonNumber
#   dim_series     — to restrict to series present in the DW (inner join)
#
# Design notes:
#   - Grain: one row per (series, season number)
#   - Hierarchy: Season < Series
#   - This dimension contains ONLY descriptive attributes — no measures
#   - avg_rating, total_votes, num_episodes belong exclusively in fact_episode_snapshot
#   - A composite unique key (tconst_series, season_number) prevents duplicates

dim_series = pd.read_parquet(f'{project_src}\\data\\silver\\dim_series.parquet')

# 1. Get unique (parentTconst, seasonNumber) pairs from title_episode
#    Drop rows with no seasonNumber — those are ungrouped specials
dim_season = (
    title_episode
    .dropna(subset=['seasonNumber'])
    [['parentTconst', 'seasonNumber']]
    .drop_duplicates()
    .rename(columns={
        'parentTconst':  'tconst_series',
        'seasonNumber':  'season_number',
    })
)

# 2. Restrict to series present in dim_series (inner join acts as filter)
dim_season = dim_season.merge(
    dim_series[['tconst']].rename(columns={'tconst': 'tconst_series'}),
    on='tconst_series',
    how='inner'
)

# 3. Build a surrogate season_id combining series and season for traceability
#    Format: <tconst>_s<season_number>  e.g. tt0032557_s1
dim_season['season_id'] = (
    dim_season['tconst_series'] + '_s' + dim_season['season_number'].astype(str)
)

# 4. Reorder columns and sort for readability
dim_season = utils.select_cols(dim_season, ['season_id', 'tconst_series', 'season_number'])
dim_season = dim_season.sort_values(['tconst_series', 'season_number']).reset_index(drop=True)

print(f"dim_season shape: {dim_season.shape}")
print(f"Unique series covered: {dim_season['tconst_series'].nunique():,}")
print(f"Sample:\n{dim_season.head(5)}")

# Write parquet file
dim_season.to_parquet(f'{project_src}\\data\\silver\\dim_season.parquet', compression='snappy', engine='pyarrow')

del dim_season
del dim_series

dim_season shape: (212376, 3)
Unique series covered: 114,501
Sample:
      season_id tconst_series  season_number
0  tt0032557_s1     tt0032557              1
1  tt0038276_s1     tt0038276              1
2  tt0039120_s1     tt0039120              1
3  tt0039122_s1     tt0039122              1
4  tt0039123_s1     tt0039123              1


# Factual tables

## participations

In [4]:
# Read dims
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')

# Convert repetitive strings to categories
dim_roles['category'] = dim_roles['category'].astype('category')
dim_roles['job'] = dim_roles['job'].astype('category')
dim_title_basic['titleType'] = dim_title_basic['titleType'].astype('category')

# If you are using Pandas 2.0+, convert IDs and text to PyArrow strings
# This is drastically more memory efficient than standard 'object' strings
# dim_roles['tconst'] = dim_roles['tconst'].astype('string[pyarrow]')

In [5]:
import gc

# Merge 1: Titles and Roles
participations_pers = dim_title_basic[['tconst','titleType', 'primaryTitle', 'genre_group_id', 'runtimeMinutes']].merge(
    dim_roles[['role_id','tconst', 'nconst', 'category', 'job', 'characters']],
    on='tconst', 
    how='left'
)

# Optional: If you no longer need the source dataframes in this script, delete them to free RAM
del dim_roles
del dim_title_basic
gc.collect() # Force Python to clear unreferenced memory immediately

# Merge 2: Add Person data
participations_pers = participations_pers.merge(
    dim_person[['nconst', 'primaryName', 'profession_group_id']], 
    on='nconst', 
    how='left'
)
del dim_person
gc.collect()

# Merge 3: Add Bridge
# Ensure bridge_kwn_titles is also pre-filtered to ONLY the columns you need before merging
participations_pers = participations_pers.merge(
    bridge_kwn_titles, 
    on=['tconst', 'nconst'], 
    how='left'
)
del bridge_kwn_titles
gc.collect()

# participations_pers = participations_pers[['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id']].copy()
participations_pers = utils.select_cols(participations_pers, ['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'role_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id'])
participations_pers['participation_count'] = 1

print(f"participations_pers.shape: {participations_pers.shape}")

# #Write parquet file
participations_pers.to_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet', compression='snappy', engine='pyarrow')

# del participations_pers

participations_pers.shape: (1133551, 14)


In [6]:
participations_pers

,nconst,primaryName,tconst,titleType,primaryTitle,runtimeMinutes,genre_group_id,role_id,category,job,characters,profession_group_id,kwn_title_group_id,participation_count
0,nm0345300,Kit Guard,tt0032557,tvSeries,The Green Archer,285.0,g_grp_29229,nm0345300_tt0032557_10,actor,NaN,"[""Dinky Stone - Henchman"",""Radio Man""]",p_grp_319897,NaN,1
1,nm0430460,Victor Jory,tt0032557,tvSeries,The Green Archer,285.0,g_grp_29229,nm0430460_tt0032557_1,actor,NaN,"[""Spike Holland""]",p_grp_398377,NaN,1
2,nm0580585,Iris Meredith,tt0032557,tvSeries,The Green Archer,285.0,g_grp_29229,nm0580585_tt0032557_2,actress,NaN,"[""Valerie Howett""]",p_grp_536568,NaN,1
3,nm0186608,James Craven,tt0032557,tvSeries,The Green Archer,285.0,g_grp_29229,nm0186608_tt0032557_3,actor,NaN,"[""Abel Bellamy""]",p_grp_173514,kwn_t_grp_1,1
4,nm0279963,Robert Fiske,tt0032557,tvSeries,The Green Archer,285.0,g_grp_29229,nm0279963_tt0032557_4,actor,NaN,"[""Savini""]",p_grp_259689,kwn_t_grp_1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1133546,nm8011539,Gleyson Spadetti,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm8011539_tt9916678_5,writer,creator,None,p_grp_7258809,kwn_t_grp_126622,1
1133547,nm10749212,Maria Fernanda Mallet,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm10749212_tt9916678_6,actress,NaN,"[""Alice"",""Lia""]",p_grp_1400479,NaN,1
1133548,nm9613196,Antonella Mattos,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm9613196_tt9916678_7,actress,NaN,"[""Bel"",""Lícia""]",p_grp_8256203,kwn_t_grp_126622,1
1133549,nm10783601,Miguel Schmidt,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm10783601_tt9916678_8,actor,NaN,"[""Lucas"",""Rafael""]",p_grp_1419278,kwn_t_grp_126622,1


## fact_series_performance

In [31]:
# Create fact_series_performance
#
# Sources:
#   title_episode  — links episodes to parent series and seasons
#   title_ratings  — averageRating and numVotes per episode
#   title_basics   — runtimeMinutes and startYear per episode
#   dim_series     — to resolve tconst -> tconst_series linkage
#   dim_season     — to resolve (tconst_series, season_number) -> season_id
#   dim_time       — to resolve startYear -> year lookup key
#   bridge_genre_group — to expand each series into one row per genre
#
# Design notes:
#   - Type: Periodic Snapshot
#   - Grain: one row per (series x season x release_year x genre)
#   - Additive measures: num_episodes, total_runtime_min, total_votes
#     can be freely summed across any dimension
#   - Semi-additive measures: avg_rating, rating_max, rating_min, stddev_rating
#     can be compared within the same series/period but MUST NOT be summed across series
#   - stddev_rating captures quality variance within a season (answers Q3: consistency)

import gc

dim_series         = pd.read_parquet(f'{project_src}\\data\\silver\\dim_series.parquet')
dim_season         = pd.read_parquet(f'{project_src}\\data\\silver\\dim_season.parquet')
dim_time           = pd.read_parquet(f'{project_src}\\data\\silver\\dim_time.parquet')
bridge_genre_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')

# 1. Attach ratings to episodes
episodes = title_episode.merge(
    title_ratings[['tconst', 'averageRating', 'numVotes']],
    on='tconst',
    how='left'
)

# Attach runtimeMinutes from title_basics (episode level)
episodes = episodes.merge(
    title_basics[['tconst', 'runtimeMinutes']],
    on='tconst',
    how='left'
)
episodes['runtimeMinutes'] = pd.to_numeric(episodes['runtimeMinutes'], errors='coerce')

# Restrict to episodes whose parent series is in dim_series
episodes = episodes.merge(
    dim_series[['tconst']].rename(columns={'tconst': 'parentTconst'}),
    on='parentTconst',
    how='inner'
)

# Drop episodes with no seasonNumber (ungrouped specials)
episodes = episodes.dropna(subset=['seasonNumber'])

# 2. Aggregate measures per (series, season)
fact = (
    episodes
    .groupby(['parentTconst', 'seasonNumber'], as_index=False)
    .agg(
        num_episodes      = ('tconst',         'count'),
        total_runtime_min = ('runtimeMinutes',  'sum'),
        total_votes       = ('numVotes',        'sum'),
        avg_rating        = ('averageRating',   'mean'),
        rating_max        = ('averageRating',   'max'),
        rating_min        = ('averageRating',   'min'),
        stddev_rating     = ('averageRating',   'std'),
    )
)

# Round semi-additive measures to match DDL precision
fact['avg_rating']    = fact['avg_rating'].round(1)
fact['rating_max']    = fact['rating_max'].round(1)
fact['rating_min']    = fact['rating_min'].round(1)
fact['stddev_rating'] = fact['stddev_rating'].round(2)

del episodes
gc.collect()

# 3. Resolve dim_season key: (parentTconst, seasonNumber) -> season_id
fact = fact.rename(columns={'parentTconst': 'tconst_series'})
fact = fact.merge(
    dim_season[['season_id', 'tconst_series', 'season_number']],
    left_on =['tconst_series', 'seasonNumber'],
    right_on=['tconst_series', 'season_number'],
    how='inner'
).drop(columns=['seasonNumber', 'season_number'])

# 4. Resolve dim_time key: startYear of the parent series -> year
series_year = dim_series[['tconst', 'start_year']].rename(columns={'tconst': 'tconst_series'})
fact = fact.merge(series_year, on='tconst_series', how='left')
fact = fact.merge(
    dim_time[['year']].rename(columns={'year': 'start_year'}),
    on='start_year',
    how='left'
)
# start_year is now the join key — rename for clarity in the final table
fact = fact.rename(columns={'start_year': 'release_year'})

# 5. Expand to one row per genre via bridge_genre_group
genre_lookup = bridge_genre_group[['tconst', 'genre_group_id', 'genre_id']].rename(
    columns={'tconst': 'tconst_series'}
)
fact = fact.merge(genre_lookup, on='tconst_series', how='left')

# 6. Select final columns and remove rows with unresolvable keys
fact = utils.select_cols(fact, [
    'tconst_series', 'season_id', 'release_year', 'genre_id',
    'num_episodes', 'total_runtime_min', 'total_votes',
    'avg_rating', 'rating_max', 'rating_min', 'stddev_rating'
])
fact = fact.dropna(subset=['release_year', 'genre_id'])
fact = fact.drop_duplicates(subset=['tconst_series', 'season_id', 'release_year', 'genre_id'])
fact = fact.reset_index(drop=True)

print(f"fact_series_performance shape: {fact.shape}")
print(f"Unique series: {fact['tconst_series'].nunique():,}")
print(f"Unique seasons: {fact['season_id'].nunique():,}")
print(f"Sample:\n{fact.head(5)}")

# Write parquet file
fact.to_parquet(f'{project_src}\\data\\silver\\fact_series_performance.parquet', compression='snappy', engine='pyarrow')

del fact
del dim_series
del dim_season
del dim_time
del bridge_genre_group
del genre_lookup
del series_year
gc.collect()

fact_series_performance shape: (299698, 11)
Unique series: 100,870
Unique seasons: 194,054
Sample:
  tconst_series     season_id  release_year genre_id  num_episodes  \
0     tt0032557  tt0032557_s1          1940    gen_1            15   
1     tt0032557  tt0032557_s1          1940    gen_3            15   
2     tt0032557  tt0032557_s1          1940    gen_7            15   
3     tt0039120  tt0039120_s1          1947   gen_10             1   
4     tt0039120  tt0039120_s1          1947   gen_13             1   

   total_runtime_min  total_votes  avg_rating  rating_max  rating_min  \
0              301.0          0.0         NaN         NaN         NaN   
1              301.0          0.0         NaN         NaN         NaN   
2              301.0          0.0         NaN         NaN         NaN   
3                0.0          0.0         NaN         NaN         NaN   
4                0.0          0.0         NaN         NaN         NaN   

   stddev_rating  
0            NaN  
1  

0